# Organize QCT POSCARs for HPC SO2

This notebook reorganizes the normal generator output from `outputs/qct_poscars/<temperature>/Ei*/POSCAR-*` into the HPC layout used for SO2 calculations:

```text
outputs/qct_poscars_hpc_SO2/Ei*/Ts*/poscars-rand-zpe/POSCAR-*
```

It also writes global `index.csv` and `index.json` files that can be used by cluster array jobs.


In [ ]:
import csv
import json
import shutil
from pathlib import Path

SOURCE_ROOT = Path("outputs/qct_poscars")
HPC_ROOT = Path("outputs/qct_poscars_hpc_SO2")

TEMPERATURE_MAP = {
    "100K": "Ts100",
    "300K": "Ts300",
    "500K": "Ts500",
}
EI_LABELS = ["Ei0.1", "Ei0.3", "Ei0.5", "Ei1", "Ei2"]
HPC_POSCAR_SUBDIR = "poscars-rand-zpe"

# Keep False by default so the generator output remains available until you decide to clean it.
CLEAN_SOURCE_AFTER_COPY = False

print(f"Source: {SOURCE_ROOT}")
print(f"HPC destination: {HPC_ROOT}")


## Copy and Reindex


In [ ]:
def _numeric_poscar_key(path: Path) -> int:
    return int(path.name.split("-")[-1])

def _copy_batch(source_dir: Path, dest_dir: Path, temperature_label: str, hpc_temperature_label: str, ei_label: str, start_job_id: int):
    if not source_dir.exists():
        raise FileNotFoundError(f"Missing source directory: {source_dir}")
    poscars = sorted(source_dir.glob("POSCAR-*"), key=_numeric_poscar_key)
    if not poscars:
        raise FileNotFoundError(f"No POSCAR-* files found in {source_dir}")

    source_metadata_path = source_dir / "metadata.json"
    source_metadata = json.loads(source_metadata_path.read_text()) if source_metadata_path.exists() else []
    metadata_by_number = {}
    for record in source_metadata:
        try:
            number = int(Path(record["poscar"]).name.split("-")[-1])
        except Exception:
            continue
        metadata_by_number[number] = record

    dest_dir.mkdir(parents=True, exist_ok=True)
    new_metadata = []
    index_rows = []
    job_id = start_job_id

    for poscar in poscars:
        number = _numeric_poscar_key(poscar)
        dest_poscar = dest_dir / poscar.name
        shutil.copy2(poscar, dest_poscar)

        record = dict(metadata_by_number.get(number, {}))
        record.update({
            "poscar": str(dest_poscar),
            "source_poscar": str(poscar),
            "surface_temperature_label": temperature_label,
            "hpc_temperature_label": hpc_temperature_label,
            "incident_energy_label": ei_label,
            "configuration": number,
            "hpc_directory": str(dest_dir),
        })
        new_metadata.append(record)

        index_rows.append({
            "job_id": job_id,
            "surface_temperature_label": temperature_label,
            "hpc_temperature_label": hpc_temperature_label,
            "incident_energy_label": ei_label,
            "incident_energy_eV": record.get("incident_energy_eV", ""),
            "configuration": number,
            "poscar": str(dest_poscar),
        })
        job_id += 1

    (dest_dir / "metadata.json").write_text(json.dumps(new_metadata, indent=2))
    return index_rows, job_id

all_index_rows = []
next_job_id = 1

for temperature_label, hpc_temperature_label in TEMPERATURE_MAP.items():
    for ei_label in EI_LABELS:
        source_dir = SOURCE_ROOT / temperature_label / ei_label
        dest_dir = HPC_ROOT / ei_label / hpc_temperature_label / HPC_POSCAR_SUBDIR
        rows, next_job_id = _copy_batch(
            source_dir=source_dir,
            dest_dir=dest_dir,
            temperature_label=temperature_label,
            hpc_temperature_label=hpc_temperature_label,
            ei_label=ei_label,
            start_job_id=next_job_id,
        )
        all_index_rows.extend(rows)
        print(f"{source_dir} -> {dest_dir}: {len(rows)} POSCAR")

(HPC_ROOT / "index.json").write_text(json.dumps(all_index_rows, indent=2))
with (HPC_ROOT / "index.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=[
        "job_id",
        "surface_temperature_label",
        "hpc_temperature_label",
        "incident_energy_label",
        "incident_energy_eV",
        "configuration",
        "poscar",
    ])
    writer.writeheader()
    writer.writerows(all_index_rows)

print(f"Wrote {len(all_index_rows)} indexed POSCAR files")
print(f"Index CSV: {HPC_ROOT / 'index.csv'}")
print(f"Index JSON: {HPC_ROOT / 'index.json'}")


## Optional Source Cleanup


In [ ]:
if CLEAN_SOURCE_AFTER_COPY:
    for child in SOURCE_ROOT.iterdir():
        if child.name == ".gitkeep":
            continue
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()
    (SOURCE_ROOT / ".gitkeep").touch(exist_ok=True)
    print(f"Cleaned {SOURCE_ROOT}, preserving .gitkeep")
else:
    print("Source cleanup skipped. Set CLEAN_SOURCE_AFTER_COPY = True to clean SOURCE_ROOT after copy.")


## HPC Layout Check


In [ ]:
expected = len(TEMPERATURE_MAP) * len(EI_LABELS)
poscar_paths = sorted(HPC_ROOT.glob(f"*/Ts*/{HPC_POSCAR_SUBDIR}/POSCAR-*"))
metadata_paths = sorted(HPC_ROOT.glob(f"*/Ts*/{HPC_POSCAR_SUBDIR}/metadata.json"))
index_rows = json.loads((HPC_ROOT / "index.json").read_text())
bad_index_paths = [row for row in index_rows if not Path(row["poscar"]).exists()]

print(f"POSCAR files: {len(poscar_paths)}")
print(f"Batch metadata files: {len(metadata_paths)} / expected {expected}")
print(f"Index rows: {len(index_rows)}")
print(f"Bad index paths: {len(bad_index_paths)}")

if bad_index_paths:
    raise ValueError("Some index rows point to missing POSCAR files")
if len(metadata_paths) != expected:
    raise ValueError("Unexpected number of batch metadata files")
if len(poscar_paths) != len(index_rows):
    raise ValueError("POSCAR count and index row count differ")

for ei_dir in sorted(path for path in HPC_ROOT.glob("Ei*") if path.is_dir()):
    counts = []
    for ts_dir in sorted(path for path in ei_dir.glob("Ts*") if path.is_dir()):
        count = len(list((ts_dir / HPC_POSCAR_SUBDIR).glob("POSCAR-*")))
        counts.append(f"{ts_dir.name}:{count}")
    print(f"{ei_dir.name}: {', '.join(counts)}")
